# Surgical CVS AI — Colab end-to-end pipeline

Run the cells **top to bottom** in a single session — this trains the whole
two-stage model from scratch with **no Google Drive needed**. Datasets, the
model cache and checkpoints all live on the runtime's local disk (`/content`).

Every cell is **idempotent**: if a cell stops but the runtime is still alive,
just re-run from the top — the repo updates in place (no re-clone), the
CholecSeg8k download is cached, and training **resumes from its last
checkpoint**. A *full* runtime reset wipes `/content`, so you start over.

**Prerequisites**
- GPU runtime: Runtime -> Change runtime type -> GPU (T4 or better).
- ~12 GB free disk for the two datasets (CholecSeg8k ~3.1 GB + Endoscapes ~5.9 GB).
- Ideally the feature branch is merged into `main`; otherwise set the branch in
  the clone cell to the feature-branch name.

In [ ]:
%cd /content
import os
if not os.path.isdir("surgical-ai"):
    !git clone -b main https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git pull --ff-only
print("working dir:", os.getcwd())

In [ ]:
# Installs dependencies and removes the stale preinstalled torchao.
!bash scripts/colab_setup.sh

## 1. Data

Both datasets download to the local disk (`/content`) — no Drive.

- **CholecSeg8k** (~3.1 GB) -> HuggingFace Hub into `./data/cholecseg8k` (cached;
  re-running this cell is cheap once cached).
- **Endoscapes2023** (~5.9 GB) -> `/content/endoscapes2023` from the CAMMA public
  mirror (`wget -c` resumes a partial download).

In [ ]:
# CholecSeg8k (~3.1 GB) -> local HuggingFace cache (cheap once cached)
!bash scripts/download_cholecseg8k.sh

# Endoscapes2023 (~5.9 GB) -> /content (downloaded each runtime; not on Drive)
!wget -q -c -O /content/endoscapes.zip https://s3.unistra.fr/camma_public/datasets/endoscapes/endoscapes.zip
!unzip -q -o /content/endoscapes.zip -d /content/endoscapes2023

## 2. Verify the data layer

Builds the video-level splits for both datasets. Non-zero frame counts
(CholecSeg8k summing to 8080) mean the loaders work.

In [ ]:
from src.data.cholecseg8k import CholecSeg8kDataset
from src.data.endoscapes import Endoscapes2023Dataset

for split in ("train", "val", "test"):
    print(f"CholecSeg8k {split:5s}: {len(CholecSeg8kDataset(split=split))} frames")
for split in ("train", "val", "test"):
    ds = Endoscapes2023Dataset(root="/content/endoscapes2023", split=split)
    print(f"Endoscapes  {split:5s}: {len(ds)} CVS keyframes")

## 3. Train the segmentation model

`train_segmentation` defaults to **SAM2 + LoRA**; the best checkpoint is
written to `outputs/<model>/best.ckpt`.

Training is **resumable** within a session — if the cell stops but the runtime
is alive, re-run it and it continues from `last.ckpt`. A full runtime *reset*
wipes `/content`, so checkpoints are lost and you restart from scratch.

- Quick pipeline check: `!python -m src.train.train_segmentation epochs=1`
- Baseline instead of SAM2: `model=unet_baseline`

In [ ]:
!python -m src.train.train_segmentation

## 4. Train the CVS classifier

Requires the trained segmentation checkpoint from step 3 and Endoscapes2023
under `/content/endoscapes2023` (downloaded in step 1). Also resumable.

In [ ]:
!python -m src.train.train_cvs data.root=/content/endoscapes2023

## 5. Benchmark + demo

`benchmark_runner` evaluates the trained checkpoints and writes
`results/benchmark_table.md`. The Gradio demo is left commented out (it starts
a blocking server).

In [ ]:
!python -m src.eval.benchmark_runner
# !python -m app.gradio_demo      # interactive demo (uncomment to launch)